# Lesson 14B: Self-Supervised Learning - Practical

Apply contrastive learning to learn representations without labels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

class SimCLR(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, projection_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection(h)
        return h, F.normalize(z, dim=-1)

def add_noise(x, noise_factor=0.3):
    """Simple augmentation: add Gaussian noise"""
    noise = torch.randn_like(x) * noise_factor
    return x + noise

def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.mm(z, z.T) / temperature
    
    mask = torch.eye(2 * batch_size, dtype=torch.bool)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    
    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ])
    
    loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
    return loss.mean()

# Load data
digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train_tensor = torch.FloatTensor(X_train)

# Train SimCLR (unsupervised)
model = SimCLR(input_dim=64, hidden_dim=128, projection_dim=64)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Training SimCLR (unsupervised)...")
for epoch in range(100):
    # Create two augmented views
    indices = torch.randint(0, len(X_train_tensor), (64,))
    x = X_train_tensor[indices]
    x_i = add_noise(x, noise_factor=0.2)
    x_j = add_noise(x, noise_factor=0.2)
    
    _, z_i = model(x_i)
    _, z_j = model(x_j)
    
    loss = nt_xent_loss(z_i, z_j, temperature=0.5)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100, Loss: {loss.item():.4f}")

# Extract learned representations
with torch.no_grad():
    h_train, _ = model(torch.FloatTensor(X_train))
    h_test, _ = model(torch.FloatTensor(X_test))
    h_train = h_train.numpy()
    h_test = h_test.numpy()

# Linear evaluation: train classifier on learned features
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(h_train, y_train)
acc = clf.score(h_test, y_test)

print(f"\n✅ Linear evaluation accuracy: {acc:.4f}")
print("SimCLR learned meaningful representations without labels!")